In [27]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import numpy as np
from datetime import datetime
import random
import os
import sys

## Load Data

In [28]:
tournaments_df = pd.read_csv('../Data/Staging/tournaments.csv').set_index(['Name', 'Year'])
players_df = pd.read_csv('../Data/Staging/players.csv').set_index('Name')
rankings_df = pd.read_csv('../Data/Staging/rankings.csv').drop(columns=["Unnamed: 0"])

matches_df = pd.read_csv('../Data/Staging/matches.csv')
matches_df = matches_df.drop(matches_df.columns[[0]], axis = 1)
# Dropping Byes
matches_df = matches_df[(matches_df['Player 1'] != 'Bye') & (matches_df['Player 2'] != 'Bye')]
matches_df

,Player 1,Player 2,Winner,Tournament Name,Round,Year
0,Lleyton Hewitt,Guillermo Canas,Lleyton Hewitt,'S-Hertogenbosch,Finals,2001
1,Lleyton Hewitt,Roger Federer,Lleyton Hewitt,'S-Hertogenbosch,Semi-Finals,2001
2,Guillermo Canas,Tommy Robredo,Guillermo Canas,'S-Hertogenbosch,Semi-Finals,2001
3,Lleyton Hewitt,Gilles Elseneer,Lleyton Hewitt,'S-Hertogenbosch,Quarter-Finals,2001
4,Roger Federer,Raemon Sluiter,Roger Federer,'S-Hertogenbosch,Quarter-Finals,2001
...,...,...,...,...,...,...
92091,Alex Bolt,Lorenzo Giustino,Alex Bolt,Zhuhai,1st Round Qualifying,2023
92092,Dominik Palan,Chukang Wang,Dominik Palan,Zhuhai,1st Round Qualifying,2023
92093,Arthur Weber,Robert Strombachs,Arthur Weber,Zhuhai,1st Round Qualifying,2023
92094,Luke Saville,Stefanos Sakellaridis,Luke Saville,Zhuhai,1st Round Qualifying,2023


A little bit of data engineering to bring in Dates to players_df, makes it easier for feature engineering

In [29]:
dates_df = tournaments_df[['Start Date','End Date', 'Surface']]
matches_df = pd.merge(left = matches_df, right = dates_df, left_on = ['Tournament Name', 'Year'], right_on = ['Name', 'Year']).drop_duplicates()
matches_df

,Player 1,Player 2,Winner,Tournament Name,Round,Year,Start Date,End Date,Surface
0,Lleyton Hewitt,Guillermo Canas,Lleyton Hewitt,'S-Hertogenbosch,Finals,2001,2001-06-18,2001-06-24,Grass
1,Lleyton Hewitt,Roger Federer,Lleyton Hewitt,'S-Hertogenbosch,Semi-Finals,2001,2001-06-18,2001-06-24,Grass
2,Guillermo Canas,Tommy Robredo,Guillermo Canas,'S-Hertogenbosch,Semi-Finals,2001,2001-06-18,2001-06-24,Grass
3,Lleyton Hewitt,Gilles Elseneer,Lleyton Hewitt,'S-Hertogenbosch,Quarter-Finals,2001,2001-06-18,2001-06-24,Grass
4,Roger Federer,Raemon Sluiter,Roger Federer,'S-Hertogenbosch,Quarter-Finals,2001,2001-06-18,2001-06-24,Grass
...,...,...,...,...,...,...,...,...,...
92182,Alex Bolt,Lorenzo Giustino,Alex Bolt,Zhuhai,1st Round Qualifying,2023,2023-09-20,2023-09-26,Hard
92183,Dominik Palan,Chukang Wang,Dominik Palan,Zhuhai,1st Round Qualifying,2023,2023-09-20,2023-09-26,Hard
92184,Arthur Weber,Robert Strombachs,Arthur Weber,Zhuhai,1st Round Qualifying,2023,2023-09-20,2023-09-26,Hard
92185,Luke Saville,Stefanos Sakellaridis,Luke Saville,Zhuhai,1st Round Qualifying,2023,2023-09-20,2023-09-26,Hard


## Feature Engineering 

### Calculating H2H

In [30]:
# TODO convert to proportions
df = matches_df.copy()

df['Start Date'] = pd.to_datetime(df['Start Date'])
df['_row_id'] = np.arange(len(df))

a = df['Player 1'].values
b = df['Player 2'].values
df['_pair'] = np.where(a <= b, a + '||' + b, b + '||' + a)

long = pd.DataFrame({
    '_pair'      : np.r_[df['_pair'].values,      df['_pair'].values],
    '_row_id'    : np.r_[df['_row_id'].values,    df['_row_id'].values],
    'Start Date' : np.r_[df['Start Date'].values, df['Start Date'].values],
    'player'     : np.r_[df['Player 1'].values,   df['Player 2'].values],
    'won'        : np.r_[
        (df['Winner'] == df['Player 1']).astype(int).values,
        (df['Winner'] == df['Player 2']).astype(int).values
    ],
    'pos'        : np.r_[np.ones(len(df), dtype=int),
                         np.full(len(df), 2, dtype=int)]
})

long = long.sort_values(['_pair', 'Start Date', '_row_id'])

long['cum_wins'] = long.groupby(['_pair', 'player'])['won'].cumsum()
long['prev_wins'] = long.groupby(['_pair', 'player'])['cum_wins'].shift(fill_value=0)

prev = (long.pivot(index='_row_id', columns='pos', values='prev_wins')
            .rename(columns={1:'Player 1 Previous Wins', 2:'Player 2 Previous Wins'}))

matches_df[['P1 Previous Wins', 'P2 Previous Wins']] = \
    prev.loc[df['_row_id']].values

matches_df.drop(columns=['_row_id','_pair'], errors='ignore', inplace=True)
matches_df.head()

,Player 1,Player 2,Winner,Tournament Name,Round,Year,Start Date,End Date,Surface,P1 Previous Wins,P2 Previous Wins
0,Lleyton Hewitt,Guillermo Canas,Lleyton Hewitt,'S-Hertogenbosch,Finals,2001,2001-06-18,2001-06-24,Grass,1,0
1,Lleyton Hewitt,Roger Federer,Lleyton Hewitt,'S-Hertogenbosch,Semi-Finals,2001,2001-06-18,2001-06-24,Grass,0,0
2,Guillermo Canas,Tommy Robredo,Guillermo Canas,'S-Hertogenbosch,Semi-Finals,2001,2001-06-18,2001-06-24,Grass,1,0
3,Lleyton Hewitt,Gilles Elseneer,Lleyton Hewitt,'S-Hertogenbosch,Quarter-Finals,2001,2001-06-18,2001-06-24,Grass,0,0
4,Roger Federer,Raemon Sluiter,Roger Federer,'S-Hertogenbosch,Quarter-Finals,2001,2001-06-18,2001-06-24,Grass,0,0


In [31]:
test_df = matches_df.copy()
p1 = 'Lleyton Hewitt'
p2 = 'Guillermo Canas'
date = datetime(2001, 6, 18)
filtered = test_df[((test_df['Player 1'] == p1) & (test_df['Player 2'] == p2)) | ((test_df['Player 2'] == p1) & (test_df['Player 1'] == p2))]
filtered = test_df[test_df['Start Date'] < date]
filtered

TypeError: '<' not supported between instances of 'str' and 'datetime.datetime'

### Calculating Player Power Index
Percentage of wins in the last 10 matches played + tournament wins

In [15]:
# --- Vectorized "previous 10 wins before this match" for each player ---
# TODO: check accuracy and also include tournament matches and convert to proportion

# 0) Copy, ensure datetime, sort by time
matches_df = matches_df.copy()
matches_df['Start Date'] = pd.to_datetime(matches_df['Start Date'])
matches_df = matches_df.sort_values(['Start Date']).reset_index(drop=True)

# 1) Stable id to merge back later
matches_df['match_id'] = np.arange(len(matches_df), dtype=np.int64)

# 2) Long form: one row per (match, player)
p1 = matches_df.rename(columns={'Player 1': 'player'})[['match_id','player','Winner','Start Date']].assign(
    is_win=lambda d: (d['Winner'] == d['player']).astype('int8'),
    side='P1'
)
p2 = matches_df.rename(columns={'Player 2': 'player'})[['match_id','player','Winner','Start Date']].assign(
    is_win=lambda d: (d['Winner'] == d['player']).astype('int8'),
    side='P2'
)
long = pd.concat([p1, p2], ignore_index=True)

# 3) Sort within player, reset index (important for clean assignment)
long = long.sort_values(['player', 'Start Date', 'match_id']).reset_index(drop=True)

# 4) Rolling wins over the PREVIOUS 10 matches (exclude current via shift)
prev = (
    long.groupby('player', sort=False, group_keys=False)
        .apply(lambda g: g['is_win'].shift(1).rolling(window=10, min_periods=1).sum())
)

long['prev10_wins'] = prev.fillna(0).astype('int16')

# 5) Map back to wide format
p1_prev = long.loc[long['side'] == 'P1', ['match_id', 'prev10_wins']] \
              .rename(columns={'prev10_wins': 'P1 Last 10 Matches'})
p2_prev = long.loc[long['side'] == 'P2', ['match_id', 'prev10_wins']] \
              .rename(columns={'prev10_wins': 'P2 Last 10 Matches'})

matches_df = (matches_df
              .merge(p1_prev, on='match_id', how='left')
              .merge(p2_prev, on='match_id', how='left')
              .drop(columns=['match_id']))

# -> matches_df now includes:
#    'P1 Last 10 Matches' and 'P2 Last 10 Matches' (int16), computed fast & correctly.
matches_df.head()

C:\Users\aman0\AppData\Local\Temp\ipykernel_4840\3261247967.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g['is_win'].shift(1).rolling(window=10, min_periods=1).sum())


,Player 1,Player 2,Winner,Tournament Name,Round,Year,Start Date,End Date,Surface,P1 Previous Wins,P2 Previous Wins,P1 Last 10 Matches,P2 Last 10 Matches
0,Tommy Haas,Nicolas Massu,Tommy Haas,Adelaide,Finals,2001,2001-01-01,2001-01-07,Hard,0,0,0,0
1,Marcelo Rios,Bohdan Ulihrach,Marcelo Rios,Doha,Finals,2001,2001-01-01,2001-01-07,Hard,0,0,0,0
2,Marcelo Rios,Vladimir Voltchkov,Marcelo Rios,Doha,Semi-Finals,2001,2001-01-01,2001-01-07,Hard,0,0,1,0
3,Bohdan Ulihrach,Nicolas Escude,Bohdan Ulihrach,Doha,Semi-Finals,2001,2001-01-01,2001-01-07,Hard,0,0,0,0
4,Bohdan Ulihrach,Yevgeny Kafelnikov,Bohdan Ulihrach,Doha,Quarter-Finals,2001,2001-01-01,2001-01-07,Hard,0,0,1,0


### Calculating Player Surface Win Rate

In [16]:
import pandas as pd
import numpy as np

# 0) Ensure datetime
matches_df['Start Date'] = pd.to_datetime(matches_df['Start Date'])

# 1) Build long once (two rows per match: one for each player)
p1 = matches_df[['Player 1','Player 2','Winner','Surface','Start Date']].copy()
p1.rename(columns={'Player 1':'player','Player 2':'opponent'}, inplace=True)

p2 = matches_df[['Player 1','Player 2','Winner','Surface','Start Date']].copy()
p2.rename(columns={'Player 2':'player','Player 1':'opponent'}, inplace=True)

long = pd.concat([p1, p2], ignore_index=True)
long['is_win'] = (long['Winner'] == long['player']).astype('int8')

# 2) Collapse to per-(player, surface, date) tallies
per_date = (
    long.groupby(['player','Surface','Start Date'], as_index=False)
        .agg(matches=('player','size'), wins=('is_win','sum'))
        .sort_values(['player','Surface','Start Date'])
)

# 3) Strictly BEFORE the date (no same-day leakage): shifted cumsums
per_date['prior_matches'] = (
    per_date.groupby(['player','Surface'])['matches'].cumsum().shift(fill_value=0)
)
per_date['prior_wins'] = (
    per_date.groupby(['player','Surface'])['wins'].cumsum().shift(fill_value=0)
)

# 4) Build fast lookup Series keyed by (player, surface, date)
key_cols = ['player','Surface','Start Date']
prior_matches_s = per_date.set_index(key_cols)['prior_matches']
prior_wins_s    = per_date.set_index(key_cols)['prior_wins']

# 5) Map onto Player 1 / Player 2 rows directly
p1_keys = list(zip(matches_df['Player 1'], matches_df['Surface'], matches_df['Start Date']))
p2_keys = list(zip(matches_df['Player 2'], matches_df['Surface'], matches_df['Start Date']))

matches_df['P1 Surface Matches'] = pd.Series(p1_keys).map(prior_matches_s).fillna(0).astype(int)
matches_df['P1 Surface Wins']    = pd.Series(p1_keys).map(prior_wins_s).fillna(0).astype(int)
matches_df['P2 Surface Matches'] = pd.Series(p2_keys).map(prior_matches_s).fillna(0).astype(int)
matches_df['P2 Surface Wins']    = pd.Series(p2_keys).map(prior_wins_s).fillna(0).astype(int)

matches_df.head()

,Player 1,Player 2,Winner,Tournament Name,Round,Year,Start Date,End Date,Surface,P1 Previous Wins,P2 Previous Wins,P1 Last 10 Matches,P2 Last 10 Matches,P1 Surface Matches,P1 Surface Wins,P2 Surface Matches,P2 Surface Wins
0,Tommy Haas,Nicolas Massu,Tommy Haas,Adelaide,Finals,2001,2001-01-01,2001-01-07,Hard,0,0,0,0,78,50,18,6
1,Marcelo Rios,Bohdan Ulihrach,Marcelo Rios,Doha,Finals,2001,2001-01-01,2001-01-07,Hard,0,0,0,0,3,2,18,7
2,Marcelo Rios,Vladimir Voltchkov,Marcelo Rios,Doha,Semi-Finals,2001,2001-01-01,2001-01-07,Hard,0,0,1,0,3,2,8,2
3,Bohdan Ulihrach,Nicolas Escude,Bohdan Ulihrach,Doha,Semi-Finals,2001,2001-01-01,2001-01-07,Hard,0,0,0,0,18,7,19,11
4,Bohdan Ulihrach,Yevgeny Kafelnikov,Bohdan Ulihrach,Doha,Quarter-Finals,2001,2001-01-01,2001-01-07,Hard,0,0,1,0,18,7,24,17


### Adding in Player Rankings

In [17]:
matches = matches_df.copy()
ranks   = rankings_df.copy()

# Coerce dtypes
matches['Start Date'] = pd.to_datetime(matches['Start Date'], errors='coerce')
ranks['Date']         = pd.to_datetime(ranks['Date'], errors='coerce')

for c in ['Player 1', 'Player 2']:
    matches[c] = matches[c].astype(str).str.strip()
ranks['Player'] = ranks['Player'].astype(str).str.strip()

# Drop bad dates
matches = matches[matches['Start Date'].notna()].copy()

# >>> CRUCIAL: sort by time key FIRST, then by group key <<<
ranks = ranks.sort_values(['Date', 'Player']).reset_index(drop=True)

orig_idx = matches.index
p1 = matches[['Start Date','Player 1']].rename(columns={'Player 1':'Player'}).assign(side='P1', _row=orig_idx)
p2 = matches[['Start Date','Player 2']].rename(columns={'Player 2':'Player'}).assign(side='P2', _row=orig_idx)
left = pd.concat([p1, p2], ignore_index=True)

# sort left by time first, then by group
left = left.sort_values(['Start Date', 'Player'], kind='mergesort').reset_index(drop=True)

merged = pd.merge_asof(
    left,
    ranks[['Player','Date','Rank']],
    left_on='Start Date',
    right_on='Date',
    by='Player',
    direction='backward',
    allow_exact_matches=False
)

wide = (merged
        .pivot(index='_row', columns='side', values='Rank')
        .reindex(orig_idx)
        .rename(columns={'P1':'P1 Ranking','P2':'P2 Ranking'}))

matches_df['P1 Ranking'] = wide['P1 Ranking'].to_numpy()
matches_df['P2 Ranking'] = wide['P2 Ranking'].to_numpy()
matches_df.head()

,Player 1,Player 2,Winner,Tournament Name,Round,Year,Start Date,End Date,Surface,P1 Previous Wins,P2 Previous Wins,P1 Last 10 Matches,P2 Last 10 Matches,P1 Surface Matches,P1 Surface Wins,P2 Surface Matches,P2 Surface Wins,P1 Ranking,P2 Ranking
0,Tommy Haas,Nicolas Massu,Tommy Haas,Adelaide,Finals,2001,2001-01-01,2001-01-07,Hard,0,0,0,0,78,50,18,6,23.0,87.0
1,Marcelo Rios,Bohdan Ulihrach,Marcelo Rios,Doha,Finals,2001,2001-01-01,2001-01-07,Hard,0,0,0,0,3,2,18,7,37.0,76.0
2,Marcelo Rios,Vladimir Voltchkov,Marcelo Rios,Doha,Semi-Finals,2001,2001-01-01,2001-01-07,Hard,0,0,1,0,3,2,8,2,37.0,46.0
3,Bohdan Ulihrach,Nicolas Escude,Bohdan Ulihrach,Doha,Semi-Finals,2001,2001-01-01,2001-01-07,Hard,0,0,0,0,18,7,19,11,76.0,48.0
4,Bohdan Ulihrach,Yevgeny Kafelnikov,Bohdan Ulihrach,Doha,Quarter-Finals,2001,2001-01-01,2001-01-07,Hard,0,0,1,0,18,7,24,17,76.0,5.0


### Randomly swapping data
Flipping Player 1 and Player 2 with probability 0.5 and keeping winner the same to create class balance and prevent overfitting

In [18]:
df = matches_df.copy()

# 50/50 mask (seed optional)
rng  = np.random.default_rng(42)
mask = rng.random(len(df)) < 0.5

# P1/P2 column pairs to swap
pairs = [
    ('Player 1', 'Player 2'),
    ('P1 Ranking', 'P2 Ranking'),
    ('P1 Previous Wins', 'P2 Previous Wins'),
    ('P1 Surface Matches', 'P2 Surface Matches'),
    ('P1 Surface Wins', 'P2 Surface Wins'),
    ('P1 Last 10 Matches', 'P2 Last 10 Matches'),
]

# Atomic swap per pair
for a, b in pairs:
    tmp = df.loc[mask, [a, b]].to_numpy()   # snapshot
    df.loc[mask, [a, b]] = tmp[:, ::-1]     # write back reversed

# Target: 0 if Player 1 won, else 1
df['Winner'] = (df['Winner'] != df['Player 1']).astype('int8')
df.head()

,Player 1,Player 2,Winner,Tournament Name,Round,Year,Start Date,End Date,Surface,P1 Previous Wins,P2 Previous Wins,P1 Last 10 Matches,P2 Last 10 Matches,P1 Surface Matches,P1 Surface Wins,P2 Surface Matches,P2 Surface Wins,P1 Ranking,P2 Ranking
0,Tommy Haas,Nicolas Massu,0,Adelaide,Finals,2001,2001-01-01,2001-01-07,Hard,0,0,0,0,78,50,18,6,23.0,87.0
1,Bohdan Ulihrach,Marcelo Rios,1,Doha,Finals,2001,2001-01-01,2001-01-07,Hard,0,0,0,0,18,7,3,2,76.0,37.0
2,Marcelo Rios,Vladimir Voltchkov,0,Doha,Semi-Finals,2001,2001-01-01,2001-01-07,Hard,0,0,1,0,3,2,8,2,37.0,46.0
3,Bohdan Ulihrach,Nicolas Escude,0,Doha,Semi-Finals,2001,2001-01-01,2001-01-07,Hard,0,0,0,0,18,7,19,11,76.0,48.0
4,Yevgeny Kafelnikov,Bohdan Ulihrach,1,Doha,Quarter-Finals,2001,2001-01-01,2001-01-07,Hard,0,0,0,1,24,17,18,7,5.0,76.0


### Converting Everything to Proportions

Using Bayesian smoothing with 50/50 prior to prevent small-sample bias in my features

In [19]:
# # ---- helper: beta-smoothed ratio with neutral fallback ----
# def beta_smoothed_ratio(wins, trials, alpha=2.0, beta=2.0, neutral=0.5):
#     wins  = np.asarray(wins)
#     trials = np.asarray(trials)
#     out = (wins + alpha) / (trials + alpha + beta)
#     return np.where(trials == 0, neutral, out)

# PRIOR_A, PRIOR_B = 2.0, 2.0  # α, β  (α=β=2 pulls harder toward 0.5 than Laplace(1,1))

# # ---- H2H (smoothed) ----
# w = df['P1 Previous Wins']
# l = df['P2 Previous Wins']
# denom = w + l
# df['H2H'] = beta_smoothed_ratio(wins=w, trials=denom, alpha=PRIOR_A, beta=PRIOR_B, neutral=0.5)

# # ---- Power Index (last 10) — smoothed instead of plain /10 ----
# # If your column is the COUNT of wins in last 10, this is correct.
# # (If it’s already a fraction, multiply by 10 first or revert to your original /10.)
# df['P1 Power Index'] = beta_smoothed_ratio(df['P1 Last 10 Matches'], 10, PRIOR_A, PRIOR_B, 0.5)
# df['P2 Power Index'] = beta_smoothed_ratio(df['P2 Last 10 Matches'], 10, PRIOR_A, PRIOR_B, 0.5)

# # ---- Surface Index (smoothed) ----
# df['P1 Surface Index'] = beta_smoothed_ratio(df['P1 Surface Wins'], df['P1 Surface Matches'],
#                                              PRIOR_A, PRIOR_B, 0.5)
# df['P2 Surface Index'] = beta_smoothed_ratio(df['P2 Surface Wins'], df['P2 Surface Matches'],
#                                              PRIOR_A, PRIOR_B, 0.5)

# # ---- Clean-up and column order ----
# df = df.drop(columns=[
#     'P1 Last 10 Matches', 'P2 Last 10 Matches',
#     'P1 Surface Wins', 'P1 Surface Matches',
#     'P2 Surface Wins', 'P2 Surface Matches',
#     'P1 Previous Wins', 'P2 Previous Wins'
# ])

# # Move 'Winner' to the end
# df = df[[c for c in df.columns if c != 'Winner'] + ['Winner']]
# df.head()

### Removing NaNs and re-ordering

In [20]:
max_rank = df[['P1 Ranking', 'P2 Ranking']].max().max()
fill_value = max_rank + 1
df[['P1 Ranking', 'P2 Ranking']] = df[['P1 Ranking', 'P2 Ranking']].fillna(fill_value)
df = df[[c for c in df.columns if c != 'Winner'] + ['Winner']]

### Saving Results

In [21]:
df.to_csv('../Data/Staging/feature_table.csv')